In [1]:
%load_ext autoreload
%autoreload 2


import genesis as gs
import logging
gs.init(logging_level=logging.WARNING, backend=gs.gpu)
from buffer import Buffer
from network import Network
from make_environment import Go2WalkingEnv


[I 03/11/26 19:04:23.603 21049846] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout
2026-03-11 19:04:25.300 Python[87019:21049846] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/3y/snyjd7qd59d_gxq1x0y5gywh0000gn/T/org.python.python.savedState


In [2]:
class Rewards:
    def __init__(self) -> None:
        pass

    def __call__(self, obs, actions, info) -> float:
        return 0


In [3]:
reward_fn = Rewards()
num_envs = 1
max_steps = 100

env = Go2WalkingEnv(
    num_envs=num_envs,
    device="mps",
    show_viewer=False,
    use_terrain=False,  # Set to True for complex terrain
    episode_length_s=20.0,
    reward_fn=reward_fn
)

[Genesis] [19:04:26] [WARNING] Viewer option 'n_rendered_envs' is deprecated and will be removed in future release. Please use 'rendered_envs_idx' instead.
[Genesis] [19:04:32] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


In [4]:
env.set_commands(lin_vel_x=1.0, lin_vel_y=0.0, ang_vel_yaw=0.0)
policy = Network(
    num_outputs=env.num_actions,
    num_inputs=env.num_obs,
    gamma=0.99,
    lmbda=0.0,
    epsilon=0.1,
)
buffer = Buffer(
    num_envs=num_envs,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=max_steps,
    device='mps'
)

In [5]:
import torch
policy.apply(lambda m: torch.nn.init.xavier_uniform_(m.weight) if hasattr(m, 'weight') else None)

Network(
  (shared): Sequential(
    (0): Linear(in_features=48, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
  )
  (actor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
  (actor_mean): Linear(in_features=32, out_features=12, bias=True)
  (critic): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [10]:
num_updates = 10
num_steps_per_update = 10
device = 'mps'
policy.to(device)

for update_idx in range(num_updates):
    obs = env.reset()
    print(obs)
    for _ in range(num_steps_per_update):

        obs = obs.to(device)
        actions, value = policy.get_actions(obs)
        log_probs, log_probs_value, entropy = policy.compute_log_probs(obs, actions)
        #print(env.device)
        #print(actions.get_device())
        #actions = actions.squeeze(0).cpu()
        #print(actions)
        obs, reward, done, info = env.step(actions)


        buffer.add_step(obs, actions, log_probs, reward, done, value)




Shapes before cat:
base_lin_vel_base: torch.Size([1, 3])
base_ang_vel_base: torch.Size([1, 3])
proj_gravity: torch.Size([1, 3])
commands: torch.Size([1, 3])
dof_pos - default: torch.Size([1, 12])
dof_vel: torch.Size([1, 12])
actions: torch.Size([1, 12])
buf_dim:  torch.Size([1, 48])
tensor([[ 0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -1.,  2.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.]], device='mps:0')
mps
0
tensor([[ 0.4340, -0.5981, -2.0526,  1.3635,  1.0561,  0.0588, -0.7485,  0.6839,
          0.2940,  0.2396, -0.5769,  0.8846]], device='mps:0')
Shapes before cat:
base_lin_vel_base: torch.Size([1, 3])
base_ang_vel_base: torch.Size([1, 3])
proj_gravity: torch.Size([1, 3])
commands: torch.Size([1, 3])
dof_pos - default: torch.Size([1, 12])
dof_vel: torch.Size([1, 12])
actions: torch.Size([1, 12])
buf_dim:  torch.Si

IndexError: index 101 is out of bounds for dimension 0 with size 101